# Quantize BiRefNet on Colab

因为你在 VS Code 中连接到的是 **远程的 Colab 运行时**，Colab 的默认工作目录（`/content`）是空的，并没有你本地计算机上的代码文件。

为了解决这个问题，我已经把整个量化流程打包到了这个 Notebook 中，它会自动在 Colab 服务器上完成所有准备工作（包括下载模型、运行量化代码），并最终把量化好的 `.onnx` 模型下载回你的电脑。

In [ ]:
# 0. 增加虚拟内存 (Swap) 防止 OOM (内存不足) 导致 Kernel 崩溃
!fallocate -l 10G /swapfile
!chmod 600 /swapfile
!mkswap /swapfile
!swapon /swapfile
print("10GB Swap memory enabled!")

# 1. 安装必要的依赖
!pip install onnx onnxruntime "Pillow<12.0" numpy rembg

In [ ]:
%%writefile download_model.py
# 2. 将下载模型的代码写为独立脚本运行，运行完立刻释放内存
import rembg
print("Downloading birefnet-general-lite model if not present...")
rembg.new_session("birefnet-general-lite")
print("Model is ready at ~/.u2net/birefnet-general-lite.onnx")

In [ ]:
# 运行下载模型的脚本
!python download_model.py

In [ ]:
%%writefile quantize_script.py
# 3. 将量化代码保存为独立脚本 (避免 Jupyter 内存泄漏导致 Kernel 崩溃)
import os
import sys
import urllib.request
from pathlib import Path
import numpy as np
from onnxruntime.quantization import (
    CalibrationDataReader,
    QuantFormat,
    QuantType,
    quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process
from PIL import Image

INPUT_SIZE = 1024
INPUT_NAME = "input_image"
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

REPO_ROOT = Path("/content")
SRC_MODEL = Path.home() / ".u2net" / "birefnet-general-lite.onnx"
DST_MODEL = REPO_ROOT / "models" / "birefnet-general-lite-int8.onnx"
LOCAL_CALIB_DIR = REPO_ROOT / "examples" / "raw"
EXTRA_CALIB_DIR = REPO_ROOT / "calib_images"

PICSUM_IDS = [
    "0", "10", "100", "1003", "1006", "1014", "1024", "1029",
    "103", "1035", "1038", "1043", "1052", "1062", "107", "108",
    "11", "111", "112", "115", "12", "120", "129", "13",
]

def fetch_calibration_extras():
    EXTRA_CALIB_DIR.mkdir(parents=True, exist_ok=True)
    downloaded = 0
    for pid in PICSUM_IDS:
        target = EXTRA_CALIB_DIR / f"picsum_{pid}.jpg"
        if target.exists():
            continue
        url = f"https://picsum.photos/id/{pid}/1024/1024.jpg"
        try:
            urllib.request.urlretrieve(url, target)
            downloaded += 1
            print(f"  downloaded {target.name}")
        except Exception as e:
            print(f"  skipped {pid}: {e}")
    if downloaded:
        print(f"Downloaded {downloaded} new calibration images")

def collect_calibration_paths():
    paths = []
    if LOCAL_CALIB_DIR.exists():
        paths.extend(sorted(LOCAL_CALIB_DIR.glob("*.png")))
        paths.extend(sorted(LOCAL_CALIB_DIR.glob("*.jpg")))
    if EXTRA_CALIB_DIR.exists():
        paths.extend(sorted(EXTRA_CALIB_DIR.glob("*.jpg")))
        paths.extend(sorted(EXTRA_CALIB_DIR.glob("*.png")))
    return paths

def preprocess(image_path: Path) -> np.ndarray:
    img = Image.open(image_path).convert("RGB")
    img = img.resize((INPUT_SIZE, INPUT_SIZE), Image.LANCZOS)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    arr = arr.transpose(2, 0, 1)[np.newaxis, :]
    arr = (arr - MEAN) / STD
    return arr.astype(np.float32)

class BirefnetCalibReader(CalibrationDataReader):
    def __init__(self, image_paths):
        self.image_paths = [p for p in image_paths if p.is_file()]
        print(f"Calibration: {len(self.image_paths)} images")
        self.idx = 0

    def get_next(self):
        if self.idx >= len(self.image_paths):
            return None
        path = self.image_paths[self.idx]
        self.idx += 1
        try:
            tensor = preprocess(path)
            print(f"  [{self.idx}/{len(self.image_paths)}] {path.name}", flush=True)
            return {INPUT_NAME: tensor}
        except Exception as e:
            print(f"  [{self.idx}/{len(self.image_paths)}] skipped {path.name}: {e}", flush=True)
            return self.get_next()

    def rewind(self):
        self.idx = 0

def main():
    if not SRC_MODEL.exists():
        print(f"❌ Source model not found at {SRC_MODEL}")
        sys.exit(1)

    DST_MODEL.parent.mkdir(parents=True, exist_ok=True)

    print("=== Step 1: Prepare calibration data ===")
    fetch_calibration_extras()
    calib_paths = collect_calibration_paths()
    calib_max = int(os.environ.get("CALIB_MAX", "4"))  # 降低到 4 以防 OOM
    if len(calib_paths) > calib_max:
        step = max(1, len(calib_paths) // calib_max)
        calib_paths = calib_paths[::step][:calib_max]
    print(f"  Total calibration images: {len(calib_paths)} (capped at CALIB_MAX={calib_max})")

    reader = BirefnetCalibReader(calib_paths)

    pre_processed = SRC_MODEL.parent / "birefnet-general-lite.preprocessed.onnx"
    print("\n=== Step 2: Pre-process model ===")
    quant_pre_process(
        input_model_path=str(SRC_MODEL),
        output_model_path=str(pre_processed),
        skip_optimization=False,
        skip_onnx_shape=False,
        skip_symbolic_shape=True,
    )

    print("\n=== Step 3: Run static quantization ===")
    quantize_static(
        str(pre_processed),
        str(DST_MODEL),
        reader,
        quant_format=QuantFormat.QDQ,
        per_channel=False,
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QInt8,
        op_types_to_quantize=["MatMul", "Conv", "Gemm"],
        reduce_range=True,
    )

    size_old = SRC_MODEL.stat().st_size // 1024 // 1024
    size_new = DST_MODEL.stat().st_size // 1024 // 1024
    print(f"\n=== Step 3: Done ===")
    print(f"  Original (FP32): {size_old} MB")
    print(f"  Quantized (INT8): {size_new} MB ({size_new / size_old * 100:.0f}% of original)")
    print(f"  Saved to: {DST_MODEL}")

if __name__ == '__main__':
    main()


In [7]:
# 执行独立脚本进行量化
!python quantize_script.py


=== Step 3: Run static quantization ===
  [1/4] picsum_0.jpg
^C


In [ ]:
# 4. 下载量化好的模型文件到你的本地电脑
from google.colab import files
files.download("/content/models/birefnet-general-lite-int8.onnx")
print("下载任务已提交，请注意浏览器弹出的下载提示。\n下载完成后，请将它移动到你本地代码的 models 目录下。")